# LLM documentation: can your tests catch a bad model card?

Exercise 3 left a registered digit classifier and no documentation. An LLM can draft the model card, but a person has to decide whether to trust the draft, and "it looks good" isn't a method. So you'll write the tests before you read a single card. Then you'll run them over cards that five tools really wrote, and find out what a model needs from your project to get the card right.

No model runs in this exercise, and you don't need an account or a key. The cards were recorded on November 7, 2026, from the frozen exercise 3 files. The document and the data are the pets, and everything that runs them is cattle, and that now includes the model.

**The short path takes about 15 minutes:** do step 1, then run its check. Steps 2 to 4 are optional and take about 20 more. Only reading? Each step shows the failure and says how to approach the fix.

## Get a terminal

You read this page and paste commands into a terminal. You don't run anything on the page. The output shown here is from a real run of these same commands on the untouched project, so it stops at the failures. The fixes are yours.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-04-llm?urlpath=lab/workspaces/exercise).** It's free and there's nothing to install. The first start can take a few minutes. You get a terminal on the left, already in the `project` folder, and this page on the right. To copy a command, select it and press Ctrl+C, or Cmd+C on a Mac. To paste it in the terminal, press Ctrl+Shift+V, or right-click and choose Paste. On a Mac, Cmd+V should paste too, but we haven't been able to test that, so use right-click if it doesn't. To edit a file, double-click it in the file panel on the far left, and save with Ctrl+S. A Binder session is temporary and has 2 GB of memory, so download anything you want to keep.

**Or use your laptop's terminal,** with this page open in your browser. It works on Linux and macOS, on x86_64 or ARM, and on Windows through WSL2. You need [conda](https://github.com/conda-forge/miniforge), and you don't need Jupyter. On Windows, keep the folder in your WSL2 home, not under `/mnt/c`. Paste this once:

```bash
git clone --branch binder-04-llm --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git llm
cd llm
conda env create --quiet --file .binder/environment.yml
conda activate repro-llm
python scripts/setup_project.py
cd project
```

Every command below runs in that `project` folder. To start over, go to the folder above it and run `python scripts/setup_project.py --reset`, which deletes your work.

## The project

`project` holds the frozen classifier's files, a prompt, starter tests, and `runs`, with one folder per recording. Each has a `card.md`, a `transcript.md` and a `manifest.yaml`. The five tools each ran once, in an empty folder with a fresh home.

| In `runs` | What it is |
|---|---|
| `claude-code`, `codex`, `cursor`, `local-model`, `opencode-free` | Real output, not edited |
| `model-a-codex-chat`, `model-b-opencode-second` | Real output for step 3 |
| `planted-wrong-accuracy`, `planted-invented-doi`, `planted-missing-version` | **Planted.** Edited by hand from Codex's card, because no tool made these errors |

Don't open the cards yet.

```bash
ls
ls runs
```

In [1]:
# Recorded output. You don't need to run this.
import sys

sys.path.insert(0, ".exercise")
from page import run

run("ls", "ls runs")

$ ls
artifacts.yaml
dvc.lock
dvc.yaml
manifest.yaml
metrics.json
params.yaml
prompts
README.md
release.yaml
run_all_cards.py
runs
score.py
sources.yaml
split.py
splits.json
tests
train.py

$ ls runs
claude-code
codex
cursor
local-model
model-a-codex-chat
model-b-opencode-second
opencode-free
planted-invented-doi
planted-missing-version
planted-wrong-accuracy
recorded-prompt.md


## Step 1: write the tests first

Say what a correct card has to contain before you see one. If you can't say what a correct result looks like, you can't test for it. A correct card:

- states the registered version that's in prod
- states the validation accuracy, to three decimals
- names the lock file that records the pipeline
- states the sizes of the three splits
- has no number and no URL that's missing from `metrics.json`, `params.yaml`, `release.yaml` or `sources.yaml`
- never claims the model is fit for clinical use

The starter test checks only that the card mentions accuracy. Run it over every card.

```bash
cat tests/test_card.py
python run_all_cards.py
```

In [2]:
# Recorded output. You don't need to run this.
run("cat tests/test_card.py", "python run_all_cards.py")

$ cat tests/test_card.py
"""Starter test. It passes on every card, so it can't tell a good card from a bad one. Replace it."""


def test_the_card_mentions_accuracy(card):
    assert "accuracy" in card.lower()

$ python run_all_cards.py
claude-code                      PASS  
codex                            PASS  
cursor                           PASS  
local-model                      PASS  
model-a-codex-chat               PASS  
model-b-opencode-second          PASS  
opencode-free                    PASS  
planted-invented-doi             PASS  
planted-missing-version          PASS  
planted-wrong-accuracy           PASS  

0 rejected.


Everything passes, including three cards that are wrong on purpose. A test that can't fail tells you nothing.

Your turn:

1. Replace the test in `tests/test_card.py` with your own, one for each rule above. `tests/conftest.py` gives your tests the card's text as `card`, the evidence files' text as `evidence`, and two helpers, `numbers(text)` and `urls(text)`, which each return a set. To try your tests on one card, set `CARD`:

```bash
CARD=runs/codex/card.md python -m pytest -q tests
```

2. Repair the prompt. `prompts/model-card.md` just says "Write a model card for this model." Rewrite it so that it asks for evidence and units: where each number comes from, and what it measures. Repairing it doesn't rerun anything. The prompt the tools actually got is in `runs/recorded-prompt.md`, and you can compare yours with it afterward.
3. Commit, and run the check. It builds a correct card from the project's files, damages it in several ways, and asks your tests about each one. So your tests have to work on cards you haven't seen. When it passes, you've finished the short path, and you can stop here.

```bash
git add tests prompts
git commit -m "Write the card tests first, and repair the prompt"
python ../check.py --short
```

On the starter project it fails like this:

In [3]:
# Recorded output. You don't need to run this.
run("python ../check.py --short")

$ python ../check.py --short
PASS  Your tests accept a correct card
FAIL  I gave your tests a card with a changed accuracy, and they accepted it. Add a test.
FAIL  I gave your tests a card with an accuracy rounded to two decimals, and they accepted it. Add a test.
FAIL  I gave your tests a card with a changed split size, and they accepted it. Add a test.
FAIL  I gave your tests a card with an extra number that isn't in the evidence, and they accepted it. Add a test.
FAIL  I gave your tests a card with a URL that isn't in the evidence, and they accepted it. Add a test.
FAIL  I gave your tests a card with no registered version, and they accepted it. Add a test.
FAIL  I gave your tests a card with no lock file named, and they accepted it. Add a test.
FAIL  I gave your tests a card with a clinical claim, and they accepted it. Add a test.
FAIL  prompts/model-card.md still doesn't ask where each number comes from, and in what unit. Repair it and commit it.

1 of 10 parts pass.


## Step 2 (optional): run the tests over the real cards

Now read the results.

```bash
python run_all_cards.py
```

For each card your tests reject, open it and find out why. Are your tests right, or too strict? If a harmless detail trips a rule, either change the rule and say why, or keep the rule and reject the card. Don't keep rerunning a model until a card passes. That's p-hacking. Keep every losing run, because someone may want to see what you rejected.

Passing isn't the same as being right. Read the cards that pass, too. A card can pass every test and still say something the evidence doesn't support, and that's a reason to write another test. Each tool's page on the site shows what our copy of the tests decided, so look there after you've made up your own mind.

## Step 3 (optional): is the method in the project, or in the chat?

Look at what the first cards say the model must not be used for. Most say "unknown", because that decision isn't anywhere in the repository. A person had decided it, in a conversation.

Model A is Codex. After its first run, it got the decisions in a chat message:

```bash
sed -n '/Chat message/,/Output/p' runs/model-a-codex-chat/transcript.md
```

In [4]:
# Recorded output. You don't need to run this.
run("sed -n '/Chat message/,/Output/p' runs/model-a-codex-chat/transcript.md")

$ sed -n '/Chat message/,/Output/p' runs/model-a-codex-chat/transcript.md
Chat message:

> Two decisions from our review, for this card. This model is for teaching only: it must not be used to make a decision about a real person or to read real documents. We refused candidate-v2, because its training rows overlap its validation and test rows. Add both to card.md.

Output:


Model B is a free model through OpenCode. It only had the repository, so it couldn't know. If Model B fails from the project alone, Model A was a hidden dependency.

Your turn:

1. Add tests to `tests/test_card.py` that require both decisions: the teaching-only limit and the refusal of `candidate-v2`. Run the cards again. Model A should pass, and `opencode-free` should fail.

```bash
python run_all_cards.py
```

2. Move the decisions out of the chat. Write them into a new file, `decisions.yaml`, in the `project` folder.
3. Read Model B's second recording, which is labeled as using the fix. It's the same tool, model, prompt and command, in a new session, and the folder also held a `decisions.yaml`. The chat from Model A never reached it. Its card should now pass your tests.

```bash
cat runs/model-b-opencode-second/manifest.yaml
CARD=runs/model-b-opencode-second/card.md python -m pytest -q tests
```

```bash
git add tests decisions.yaml
git commit -m "Move the decisions into the project"
```

## Step 4 (optional): record what you accepted

"We used the chatbot" isn't a methods statement. Here's a real manifest from one of the runs. `not_exposed` says what the provider doesn't tell you, and that's part of the record.

```bash
cat runs/codex/manifest.yaml
```

In [5]:
# Recorded output. You don't need to run this.
run("cat runs/codex/manifest.yaml")

$ cat runs/codex/manifest.yaml
# Filled in from the recording. Nothing in card.md was edited.
tool: Codex CLI
tool_version: 0.165.0
provider: OpenAI
model: gpt-6.1-sol (the default for this account)
revision: not exposed beyond the model name
run_date: 2026-11-07T19:11:17Z
settings:
  temperature: not exposed
  seed: not exposed
  max_tokens: not exposed
prompt: runs/recorded-prompt.md
command: "codex exec --skip-git-repo-check --sandbox workspace-write --json \"$(cat prompt.md)\""
evaluation: tests/test_card.py
not_exposed:
  - sampling settings and seed
  - the tool's built-in instructions
  - any serving changes behind the model name


Your turn:

1. Pick the card you accept. Fill in `manifest.yaml` for that run, including `acceptance`, with your test result, and `reviewer`, with who decided. If a setting isn't exposed, say so.
2. Write `decisions.md`. Name the run as it's named under `runs`, and say why you accepted it. Don't delete the others.

```bash
git add manifest.yaml decisions.md
git commit -m "Accept a card and record why"
```

## Check everything

This is the full check, for steps 1 to 4. It works in a throwaway clone, so your project is safe.

```bash
python ../check.py
```

On the starter project it fails like this:

In [6]:
# Recorded output. You don't need to run this.
run("python ../check.py")

$ python ../check.py
PASS  Your tests accept a correct card
FAIL  I gave your tests a card with a changed accuracy, and they accepted it. Add a test.
FAIL  I gave your tests a card with an accuracy rounded to two decimals, and they accepted it. Add a test.
FAIL  I gave your tests a card with a changed split size, and they accepted it. Add a test.
FAIL  I gave your tests a card with an extra number that isn't in the evidence, and they accepted it. Add a test.
FAIL  I gave your tests a card with a URL that isn't in the evidence, and they accepted it. Add a test.
FAIL  I gave your tests a card with no registered version, and they accepted it. Add a test.
FAIL  I gave your tests a card with no lock file named, and they accepted it. Add a test.
FAIL  I gave your tests a card with a clinical claim, and they accepted it. Add a test.
FAIL  prompts/model-card.md still doesn't ask where each number comes from, and in what unit. Repair it and commit it.
FAIL  Your tests accept these planted cards

## On your own project

Next time an LLM writes something you'd sign, write down first what a correct result must contain and what it must never do. Turn that into tests. Keep the decisions in the project, not the chat, and record the model and its conditions in a manifest. Then give the same task to a second model. If it can't do the work from the project alone, the first model's chat was part of your method.

## About the data and the recordings

The classifier and its files come from exercise 3, frozen at the tag `exercise-3-frozen`, with `release.yaml` and `sources.yaml` added to state the registry's contents and the data's source. The digits data is Alpaydin, E. and Kaynak, C. (1998), Optical Recognition of Handwritten Digits, UCI Machine Learning Repository, doi:10.24432/C50P49, published under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

Each `card.md` from a tool is exactly what that tool wrote. Each `transcript.md` is trimmed to the task. The planted cards say so in their manifests.